<a href="https://colab.research.google.com/github/lqi234488/AICUP-2026-Table-Tennis-Prediction/blob/main/notebooks/%E5%9F%BA%E6%96%BC%E6%99%82%E5%BA%8F%E8%B3%87%E6%96%99%E4%B9%8B%E6%A1%8C%E7%90%83%E6%88%B0%E8%A1%93%E8%88%87%E7%B5%90%E6%9E%9C%E9%A0%90%E6%B8%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

基於時序資料之桌球戰術與結果預測競賽

In [1]:
import pandas as pd
import numpy as np

# 1. 讀取資料
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 核心特徵選取
# 這些是每一拍會變動的特徵 (Time-step features)
features = ['strikeId', 'handId', 'strengthId', 'spinId', 'positionId', 'actionId']
target = ['actionId','pointId','serverGetPoint'] # 假設我們要預測下一拍的動作

# 3. 依照 rally_uid 進行分組轉換
def create_sequences(df, max_len=15):
    sequences = []
    labels = []

    for uid, group in df.groupby('rally_uid'):
        # 確保依擊球順序排列
        group = group.sort_values('strikeNumber')

        # 提取特徵矩陣
        feat_matrix = group[features].values

        # 建立滑動窗口 (例如用前 3 拍預測第 4 拍)
        for i in range(1, len(feat_matrix)):
            seq = feat_matrix[:i] # 前 i 拍
            # 補零到固定長度 max_len
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), len(features)))
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            sequences.append(seq)
            labels.append(group.iloc[i][target]) # 第 i+1 拍的動作

    return np.array(sequences), np.array(labels)

# 執行轉換 (這步會花一點時間)
# X_train, y_train = create_sequences(train_df)

檢查

In [2]:
import pandas as pd
import os

print("目前目錄下的檔案有：", os.listdir('/content/'))
# 1. 自動檢查 Colab 目前有哪些檔案
current_files = os.listdir('/content/')
print(f"目前目錄下的檔案：{current_files}")

# 2. 嘗試用正確的路徑讀取 (加上 /content/ 前綴)
try:
    # 這裡確保路徑是 Colab 的預設路徑
    train = pd.read_csv('/content/train.csv')
    print("✅ 成功讀取 train.csv！")

    # 3. 執行你剛才想跑的分析
    print("\n--- 每一個回合(Rally)的拍數統計 ---")
    strike_counts = train.groupby('rally_uid')['strikeNumber'].max()
    print(strike_counts.value_counts().sort_index())

except FileNotFoundError:
    print("❌ 錯誤：找不到檔案！請確認左側資料夾選單中，檔名是否真的叫 'train.csv'")
except Exception as e:
    print(f"❌ 發生其他錯誤：{e}")

目前目錄下的檔案有： ['.config', 'train.csv', 'test.csv', 'sample_data']
目前目錄下的檔案：['.config', 'train.csv', 'test.csv', 'sample_data']
✅ 成功讀取 train.csv！

--- 每一個回合(Rally)的拍數統計 ---
strikeNumber
2     1869
3     2585
4     3030
5     1933
6     1598
7      974
8      793
9      520
10     379
11     287
12     221
13     158
14     135
15      93
16      65
17      55
18      55
19      46
20      22
21      26
22      18
23      27
24      20
25       7
26      12
27       8
28       4
29       8
30       5
31       3
32       5
33       4
34       2
35       6
36       3
37       4
38       2
39       2
40       5
42       3
45       1
46       1
52       1
Name: count, dtype: int64


In [3]:
# 範例：將資料按 rally 排序
train = train.sort_values(['rally_uid', 'strikeNumber'])

# 找出所有的類別數量 (用於 Embedding)
num_actions = train['actionId'].nunique()
num_positions = train['positionId'].nunique()
num_getpoint = train['serverGetPoint'].nunique()

print(f"總共有 {num_actions} 種動作，{num_positions} 種落點方位。{num_getpoint} 種得分")

總共有 19 種動作，4 種落點方位。2 種得分


前處理

In [4]:
import pandas as pd

def engineering_features(df):
    # 1. 數值型歸一化 (讓數值落在 0~1 之間，對 LSTM 收斂很有幫助)
    df['strengthId'] = df['strengthId'] / 3.0
    df['scoreSelf'] = df['scoreSelf'] / 15.0
    df['scoreOther'] = df['scoreOther'] / 15.0
    df['strikeNumber'] = df['strikeNumber'] / 52.0 # 假設一回合最多30拍

    # 2. 類別型 One-hot Encoding
    # 我們只針對純類別進行展開
    df = pd.get_dummies(df, columns=['handId', 'spinId', 'positionId'])

    return df

# 執行轉換
train_df_eng = engineering_features(train)
test_df_eng = engineering_features(test)

# 【重要】對齊測試集與訓練集的欄位 (避免 One-hot 後欄位數量不一致)
train_df_eng, test_df_eng = train_df_eng.align(test_df_eng, join='left', axis=1, fill_value=0)

# 找出現在所有的特徵欄位名稱 (扣除標籤和 UID)
# 排除掉那三個 Output 標籤
exclude_cols = ['rally_uid', 'actionId', 'pointId', 'serverGetPoint']
NEW_FEATURE_COLS = [c for c in train_df_eng.columns if c not in exclude_cols]

print(f"新的特徵數量：{len(NEW_FEATURE_COLS)}")

新的特徵數量：24


第一步：prepare_data 構建訓練資料 (X, y)
我們需要把 train.csv 的每一列轉成「序列」，並提取每個 rally_uid 的最終結果作為標籤。

In [19]:
import numpy as np

def prepare_train_data_v9(df, feature_list, max_len=10):
    X, y_act, y_ptr, y_srv = [], [], [], []

    # 1. 保險：移除重複的欄位名，確保 feature_list 乾淨
    df = df.loc[:, ~df.columns.duplicated()]
    feature_list = list(dict.fromkeys(feature_list))
    n_features = len(feature_list)

    # 2. 以每一回合 (Rally) 為單位進行處理
    for uid, group in df.groupby('rally_uid'):
        # 確保時序正確
        group = group.sort_values('strikeNumber')

        # 提取這組 Rally 的所有 24 個特徵
        feat_matrix = group[feature_list].values.astype('float32')

        # 滑動窗口：每一拍產生一個訓練樣本
        for i in range(1, len(feat_matrix)):
            # 取出前 i 拍作為 X
            seq = feat_matrix[:i]

            # 補齊長度 (Padding)
            if len(seq) < max_len:
                pad = np.zeros((max_len - len(seq), n_features), dtype='float32')
                seq = np.vstack((pad, seq))
            else:
                seq = seq[-max_len:]

            X.append(seq)

            # 對應的答案是第 i 拍的內容
            y_act.append(group.iloc[i]['actionId'])
            y_ptr.append(group.iloc[i]['pointId'])
            y_srv.append(group.iloc[i]['serverGetPoint'])

    return np.array(X), np.array(y_act), np.array(y_ptr), np.array(y_srv)

# --- 重新生成訓練集 ---
X_train_final, y_act, y_ptr, y_srv = prepare_train_data_v9(train_df_eng, NEW_FEATURE_COLS)

print(f"訓練集準備完成！")
print(f"X_train_final 形狀: {X_train_final.shape}") # 應該會是 (樣本數, 10, 24)

訓練集準備完成！
X_train_final 形狀: (69712, 10, 24)


prepare_test_data

In [18]:
import numpy as np

def prepare_test_data_v8(df, feature_list, max_len=10):
    X_list = []
    uids = []

    # 確保 feature_list 裡面沒有重複的名稱
    feature_list = list(dict.fromkeys(feature_list))
    n_features = len(feature_list)

    for uid, group in df.groupby('rally_uid'):
        # 排序確保時序正確
        group = group.sort_values('strikeNumber')

        # 只選取特徵，並強制轉為 float32 避免 Object 型態污染
        # .loc[:, ~group.columns.duplicated()] 確保不會選到重複的欄位
        clean_group = group.loc[:, ~group.columns.duplicated()]
        feat_matrix = clean_group[feature_list].values.astype('float32')

        # 測試集邏輯：我們只需要「最後現有的狀態」來預測下一球
        seq = feat_matrix

        if len(seq) < max_len:
            pad = np.zeros((max_len - len(seq), n_features), dtype='float32')
            seq = np.vstack((pad, seq))
        else:
            seq = seq[-max_len:]

        X_list.append(seq)
        uids.append(uid)

    # 最後轉換成 3D 陣列 (樣本數, 10, 24)
    return np.array(X_list), uids

# --- 正式執行 ---
# 確保使用的是 _eng 結尾的處理後資料
X_test_final, test_uids_final = prepare_test_data_v8(test_df_eng, NEW_FEATURE_COLS)

print(test_df_eng['rally_uid'].nunique())
print(f"✅ 測試集準備完成！")
print(f"X_test_final 形狀: {X_test_final.shape}") # 應該會是 (2353, 10, 24)


1236
✅ 測試集準備完成！
X_test_final 形狀: (1236, 10, 24)


第二步：建立多輸出模型 (Keras Functional API)
這段程式碼會建立一個 LSTM 模型，最後分出三個輸出層：

In [22]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout

# 1. 動態設定輸入維度 (現在是 24)
n_features = len(NEW_FEATURE_COLS)

# 2. 定義模型架構
inputs = Input(shape=(10, n_features), name='main_input')

# 共享的 LSTM 層
x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.2)(x) # 加入一點 Dropout 防止過擬合
x = LSTM(32)(x)
x = Dropout(0.2)(x)

# 三個獨立的輸出頭
# 動作預測 (假設有 17 種動作)
act_head = Dense(17, activation='softmax', name='act_head')(x)
# 落點預測 (0-10 區)
ptr_head = Dense(11, activation='softmax', name='ptr_head')(x)
# 是否得分 (二元分類)
srv_head = Dense(1, activation='sigmoid', name='srv_head')(x)

# 3. 組裝模型
model_multi_v2 = Model(inputs=inputs, outputs=[act_head, ptr_head, srv_head])

# 4. 編譯模型 (加入你之前調好的 Loss Weights)
model_multi_v2.compile(
    optimizer='adam',
    loss={
        'act_head': 'sparse_categorical_crossentropy',
        'ptr_head': 'sparse_categorical_crossentropy',
        'srv_head': 'binary_crossentropy'
    },
    metrics={'act_head': 'accuracy', 'ptr_head': 'accuracy', 'srv_head': 'accuracy'},
    loss_weights={'act_head': 1.0, 'ptr_head': 1.2, 'srv_head': 0.8}
)

print(f"✅ 模型重新建立完成！輸入維度已更新為: (10, {n_features})")


✅ 模型重新建立完成！輸入維度已更新為: (10, 24)


重新訓練

In [23]:
history = model_multi_v2.fit(
    X_train_final,
    {'act_head': y_act, 'ptr_head': y_ptr, 'srv_head': y_srv},
    validation_split=0.2, # 這裡也可以直接用 validation_split 讓 Keras 幫你切
    epochs=30,
    batch_size=64
)

Epoch 1/30
872/872 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - act_head_accuracy: 0.2644 - act_head_loss: 2.1729 - loss: 5.0427 - ptr_head_accuracy: 0.2418 - ptr_head_loss: 1.9266 - srv_head_accuracy: 0.5147 - srv_head_loss: 0.6968 - val_act_head_accuracy: 0.2688 - val_act_head_loss: 2.1197 - val_loss: 4.9452 - val_ptr_head_accuracy: 0.2614 - val_ptr_head_loss: 1.8934 - val_srv_head_accuracy: 0.5246 - val_srv_head_loss: 0.6917
Epoch 2/30
872/872 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - act_head_accuracy: 0.3216 - act_head_loss: 1.9654 - loss: 4.7624 - ptr_head_accuracy: 0.2632 - ptr_head_loss: 1.8684 - srv_head_accuracy: 0.5144 - srv_head_loss: 0.6940 - val_act_head_accuracy: 0.3358 - val_act_head_loss: 1.9324 - val_loss: 4.7336 - val_ptr_head_accuracy: 0.2785 - val_ptr_head_loss: 1.8727 - val_srv_head_accuracy: 0.5259 - val_srv_head_loss: 0.6923
Epoch 3/30
872/872 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - act_head_accuracy: 0.3703 - act_head_loss: 1.8235 - loss: 4.5889 - ptr_head_accuracy: 0.2769 - p

進行預測

In [25]:
import numpy as np
import pandas as pd

# 1. 進行預測 (會得到三個輸出的機率分佈)
predictions = model_multi_v2.predict(X_test_final)

# 2. 轉換結果：將機率轉為具體的類別 ID
# 假設你的輸出順序是 [act_head, ptr_head, srv_head]
pred_actions = np.argmax(predictions[0], axis=1) # 擊球動作 (0-16)
pred_points = np.argmax(predictions[1], axis=1)  # 落點區域 (0-10)
pred_servers = (predictions[2] > 0.5).astype(int).flatten() # 是否得分 (0 或 1)

print(f"預測完成！樣本數：{len(pred_actions)}")

39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
預測完成！樣本數：1236


In [28]:
#DEBUG
# 1. 原始測試集有多少回合？
print("原始 test.csv 回合數:", test['rally_uid'].nunique())

# 2. 做完工程後的測試集有多少回合？
print("工程後 test_df_eng 回合數:", test_df_eng['rally_uid'].nunique())

# 3. 檢查看看是不是有空的資料
print("測試集是否有空值:", test_df_eng[NEW_FEATURE_COLS].isnull().sum().sum())

原始 test.csv 回合數: 1236
工程後 test_df_eng 回合數: 1236
測試集是否有空值: 0


建立並匯出 Submission 檔案

In [37]:
# 1. 直接把你確認有數字的那個 DataFrame 存起來
# 確保欄位順序正確
submission_1236 = df_debug[['rally_uid', 'actionId', 'pointId', 'serverGetPoint']]

# 2. 強制轉成整數（去掉 .0）
submission_1236 = submission_1236.astype({'actionId': int, 'pointId': int, 'serverGetPoint': int})

# 3. 直接存檔
submission_1236.to_csv('my_submission.csv', index=False)

print(f"✅ 檔案已產出：my_submission.csv (共 {len(submission_1236)} 筆)")

✅ 檔案已產出：my_submission.csv (共 1236 筆)
